In [ ]:
# TODO Add notebook documentation and description.
# Uses studio or snotel_metloom conda envs
# Need to update output dirs for saving figures to sk3/jmhu/thp dir rather than straight to public_html for viewing

In [ ]:
import sys
import os
import re

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import copy

sys.path.append('/uufs/chpc.utah.edu/common/home/u6058223/git_dirs/env/')
import helpers as h

import seaborn as sns
sns.set_palette('icefire')

In [ ]:
def prep_isnobal_dfs(basin, thisvar, runnames = ['baseline', 'hrrr_spires', 'unified'], og_work_dir=None):
    # Directories
    if og_work_dir is None:
        og_work_dir = f'/uufs/chpc.utah.edu/common/home/skiles-group3/model_runs/{basin}_100m_isnobal/'
    thp_work_dir = f'/uufs/chpc.utah.edu/common/home/skiles-group2/model_runs_jmh/thp/{basin}_hrrr_radiation'
    isnobal_dfs = []
    # Use runnames to determine the runtype matching
    if 'baseline' in runnames:
        # iSnobal files -- period of record approach
        baseline = h.fn_list(og_work_dir, f'wy*/*iSnobal-HRRR_{thisvar}*.csv')
        # Load files into por dfs per run type
        baseline_dfs = [pd.read_csv(f) for f in baseline]
        # Convert the Date column to datetime dtype
        baseline_dfs = convert_datetime_dtype(baseline_dfs)
        # Concatenate the dfs for each run type
        baseline_df = pd.concat(baseline_dfs, ignore_index=True)
        # check the shapes of the dfs
        print(f"Baseline df shape: {baseline_df.shape}")
        # Modify the column names to be {runname}_Date
        baseline_df = modify_dt_colnames([baseline_df], ['baseline'])[0]
        # set the datetime column as the index for each df
        baseline_df.set_index('baseline_date', inplace=True)
        isnobal_dfs.extend([baseline_df])

    if 'hrrr_spires' in runnames:
        hrrr_spires = h.fn_list(og_work_dir, f'wy*/*HRRR-SPIReS_{thisvar}*.csv')
        hrrr_spires_dfs = [pd.read_csv(f) for f in hrrr_spires]
        hrrr_spires_dfs = convert_datetime_dtype(hrrr_spires_dfs)
        hrrr_spires_df = pd.concat(hrrr_spires_dfs, ignore_index=True)
        print(f"HRRR-SPIReS df shape: {hrrr_spires_df.shape}")
        hrrr_spires_df = modify_dt_colnames([hrrr_spires_df], ['hrrr_spires'])[0]
        hrrr_spires_df.set_index('hrrr_spires_date', inplace=True)
        isnobal_dfs.extend([hrrr_spires_df])

    if 'unified' in runnames:
        unified = h.fn_list(thp_work_dir, f'wy*/*{thisvar}*.csv')
        unified_dfs = [pd.read_csv(f) for f in unified]
        # Process unified df to drop the 00:00:00 time component from the datetime column
        for i, unified_df in enumerate(unified_dfs):
            unified_df = drop_time_from_datetime(unified_df, dt_col_idx=0, time2drop='00:00:00')
            unified_dfs[i] = unified_df
        unified_dfs = convert_datetime_dtype(unified_dfs)
        unified_df = pd.concat(unified_dfs, ignore_index=True)
        print(f"Unified df shape: {unified_df.shape}")
        unified_df = modify_dt_colnames([unified_df], ['unified'])[0]
        unified_df.set_index('unified_date', inplace=True)
        isnobal_dfs.extend([unified_df])
    return isnobal_dfs

def convert_datetime_dtype(df_list, verbose=False):
    """
    Changes all object columns in dataframe list to datetime, if possible.
    Otherwise, it leaves the column as is and prints a message.
    """
    if verbose:
        # show all the dtypes of the columns in each DataFrame
        for i, df in enumerate(df_list):
            print(f"DataFrame {i} dtypes:")
            print(df.dtypes)
            print("\n")

    # Convert the object columns to datetime
    for i, df in enumerate(df_list):
        for col in df.columns:
            if df[col].dtype == 'object':
                try:
                    df[col] = pd.to_datetime(df[col])
                    if verbose: print(f"Converted column '{col}' in DataFrame {i} to datetime.")
                except Exception as e:
                    print(f"Could not convert column '{col}' in DataFrame {i} to datetime: {e}")
    return df_list

def drop_time_from_datetime(df, dt_col_idx, time2drop='00:00:00'):
    # Drop the rows that contain the 00:00:00 hour runs
    df = df[~df.iloc[:, dt_col_idx].str.contains(time2drop)]
    # Reset the index of the DataFrame
    df.reset_index(drop=True, inplace=True)
    return df

def modify_dt_colnames(df_list, colnames):
    for df, colname in zip(df_list, colnames):
        df.rename(columns={df.columns[0]: f'{colname}_date'}, inplace=True)
    return df_list

def process_snotel_por(df, rows2skip=65):
    # Pull in SNOTEL PoR data
    site_nums = df.columns.str.extract(r'((?<=\()\d+)').astype(int).values.flatten()
    print(site_nums)
    # Pull in all the csvs in this order that match at least that site
    snotel_dir = '/uufs/chpc.utah.edu/common/home/skiles-group3/ancillary_sdswe_products/SNOTEL'
    snotel_fns = []
    for site in site_nums:
        site_fns = h.fn_list(snotel_dir, f'*{site}*.csv')
        # this way, no need to index with fn_list, can just extend the entire list
        snotel_fns.extend(site_fns)
    # Read them into a big snotel_df list
    snotel_dfs = [pd.read_csv(fn, skiprows=rows2skip) for fn in snotel_fns]

    # Drop all columns from the df except for the date and snow depth
    for i, df in enumerate(snotel_dfs):
        df = df.iloc[:, [0, 1]]
        print(df.columns)
        # Set the date column as the index
        df.set_index(df.columns[0], inplace=True)
        snotel_dfs[i] = df

    # Note this is all in centimeters! needs converting for the plotting
    # Concatenate all the snotel_dfs into one big df using the date as the index
    snotel_df = pd.concat(snotel_dfs, axis=1)
    # convert the date index to datetime
    snotel_df.index = pd.to_datetime(snotel_df.index)
    # Arrange the index such that it is chronological
    snotel_df.sort_index(inplace=True)
    return snotel_df

def plot_snotel_comparison(basin, thisvar, snotel_df,
                           isnobal_dfs,
                           runnames = ['baseline', 'hrrr_spires', 'unified'],
                           colors = ['gold', 'firebrick', 'dodgerblue']):
    # Loop through the columns (each snotel site) and plot the time series for each df on the same subplot
    site_names = isnobal_dfs[0].columns
    unified_df = isnobal_dfs[-1]
    total_sites = len(site_names)
    fig, axa = plt.subplots(nrows=total_sites, ncols=1, figsize=(8, 2*total_sites), sharex=True, sharey=True)
    for col, ax in zip (site_names, axa.flatten()):
        # Subset the SNOTEL data for this site (parse by site number in column name) and date range
        snotel_site_subdf = snotel_df[(snotel_df.index >= unified_df.index.min()) & (snotel_df.index <= unified_df.index.max())]
        # Keep only columns that contain the site number or the date
        col_slice = [c for c in snotel_site_subdf.columns if col in c]
        snotel_site_subdf = snotel_site_subdf[col_slice]
        # Plot SNOTEL, modifying for cm to m plotting
        ax.plot(snotel_site_subdf/100, label='SNOTEL', color='lightgrey', linestyle='--', lw=1, marker='.', markersize=2)
        for df, runname, color in zip(isnobal_dfs, runnames, colors):
            ax.plot(df[col], label=runname, color=color)
        # Plot the site name within the subplot
        ax.annotate(col, xy=(0.05, 0.9), xycoords='axes fraction', fontsize=10, fontstyle='italic')
        # Set xlims to be 1 day beyond the unified_df index range
        ax.set_xlim([unified_df.index.min() - pd.Timedelta(days=1), unified_df.index.max() + pd.Timedelta(days=1)])
        ax.set_ylabel('Snow Thickness (m)')
        # Make the legend horizontal, 1 row, at the top center of the subplot, just for the first subplot
        if col == site_names[0]:
            ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.25), ncol=len(runnames)+1)
    plt.xlabel('Date')
    plt.suptitle(f'{basin.capitalize()} iSnobal Snow Depth at Snotel Sites', y=1.01)
    plt.tight_layout()

    # Save the figure
    outdir = '/uufs/chpc.utah.edu/common/home/u6058223/public_html/thp_update'
    outname = f'{outdir}/{basin}/{basin}_snotel_por_{thisvar}_comparison.png'
    print(f'Saving to {outname}')
    plt.savefig(outname, dpi=300, bbox_inches='tight')

# SNOTEL POR iSnobal comparison plots

In [ ]:
basin = 'yampa'
thisvar = 'thickness'
baseline_df, hrrr_spires_df, unified_df = prep_isnobal_dfs(basin, thisvar, runnames=['baseline', 'hrrr_spires', 'unified'])
snotel_df = process_snotel_por(df=unified_df)
plot_snotel_comparison(basin, thisvar, snotel_df,
                       isnobal_dfs = [baseline_df, hrrr_spires_df, unified_df],
                       runnames = ['baseline', 'hrrr_spires', 'unified'],
                       colors = ['gold', 'firebrick', 'dodgerblue'])

### This with the animas basin to check out the baselin runs and og_work_dir

In [ ]:
basin = 'animas'
thisvar = 'thickness'
baseline_df, hrrr_spires_df, unified_df = prep_isnobal_dfs(basin, thisvar,
                                                            og_work_dir='/uufs/chpc.utah.edu/common/home/skiles-group2/model_runs_jmh/_animas_100m_isnobal_toarchive')
snotel_df = process_snotel_por(df=unified_df)
plot_snotel_comparison(basin, thisvar, snotel_df,
                       isnobal_dfs = [baseline_df, hrrr_spires_df, unified_df],
                       runnames = ['baseline', 'hrrr_spires', 'unified'],
                       colors = ['gold', 'firebrick', 'dodgerblue'])

### Plot just the unified run against SNOTEL

In [ ]:
unified_df = prep_isnobal_dfs(basin, thisvar, runnames=['unified'])[0]
snotel_df = process_snotel_por(df=unified_df)
plot_snotel_comparison(basin, thisvar, snotel_df,
                       isnobal_dfs = [unified_df],
                       runnames = ['unified'],
                       colors = ['dodgerblue'])

# Compute diagnostic stats

### Statistics functions

In [ ]:
# Pulled from time_series_por.ipynb
# Moved here to avoid issues with metloom pulling down SNOTEL PoR
# Using direct pulls with wget and PoR csvs

def calc_lrm_stats(model, obs):
    # Compute a line of best fit
    m, b = np.polyfit(obs, model, 1)

    # Add the fit equation to the plot
    fit_eq = f'y = {m:.2f}x + {b:.2f}'
    # Add the correlation coefficient
    r = np.corrcoef(obs, model)[0, 1]
    # fit_eq += f'\nr = {r:.2f}'

    # Add the coefficient of determination R2
    r_squared = r ** 2

    # Add the number of points
    n_points = len(obs)

    return r, r_squared, n_points, fit_eq, m

def calculate_kge(r, model, obs):
    '''Calculate Kling-Gupta Efficiency (KGE) between two depth arrays'''
    # KGE is calculated as:
    # KGE = 1 - sqrt((r - 1)^2 + (alpha - 1)^2 + (beta - 1)^2)
    # where r is the correlation coefficient, alpha is the ratio of the standard deviations,
    # and beta is the ratio of the means.
    kge = 1 - np.sqrt((r - 1) ** 2 + (np.std(model) / np.std(obs) - 1) ** 2 + (np.mean(model) / np.mean(obs) - 1) ** 2)
    return kge

# Calculate RMSE first
def calculate_rmse(model, obs):
    '''Calculate RMSE between two depth arrays'''
    rmse = np.sqrt(np.nanmean((model - obs) ** 2))
    return rmse

# Now based on a defined normalization factor, calculate the normalized RMSE
def calculate_nrmse(model, obs, normalization_factor):
    '''Calculate normalized RMSE between two depth arrays
    Normalization factors: range of ASO depth, mean ASO depth, standard deviation of ASO depth
    '''
    rmse = calculate_rmse(model, obs)
    if normalization_factor == 'range':
        normalization_factor = np.nanmax(obs) - np.nanmin(obs)
    elif normalization_factor == 'mean':
        normalization_factor = np.nanmean(obs)
    elif normalization_factor == 'std':
        normalization_factor = np.nanstd(obs)
    nrmse = rmse / normalization_factor
    return nrmse

def calculate_mae(model, obs):
    '''Calculate Mean Absolute Error between two depth arrays'''
    mae = np.nanmean(np.abs(model - obs))
    return mae

def remove_empty_dates(model, obs):
    '''Removes corresponding entries missing in either model or obs'''
    # Identify dates where model is nan
    model_nan_dates = model[model.isna()].index
    # Identify dates where obs is nan
    obs_nan_dates = obs[obs.isna()].index
    # Combine the two sets of dates
    nan_dates = model_nan_dates.union(obs_nan_dates)
    # Remove the nan dates from both model and obs
    model_cleaned = model.drop(nan_dates)
    obs_cleaned = obs.drop(nan_dates)
    # Remove any dates where the model value is infinite
    model_infinite_dates = model_cleaned[model_cleaned.isin([np.inf, -np.inf])].index
    obs_infinite_dates = obs_cleaned[obs_cleaned.isin([np.inf, -np.inf])].index
    # Combine the two sets of infinite dates
    infinite_dates = model_infinite_dates.union(obs_infinite_dates)
    # Remove the infinite dates from both model and obs
    model_cleaned = model_cleaned.drop(infinite_dates)
    obs_cleaned = obs_cleaned.drop(infinite_dates)
    return model_cleaned, obs_cleaned

def trim_series(model, obs):
    '''Trim the model and observation series to the same length'''
    start_date = max(model.index.min(), obs.index.min())
    end_date = min(model.index.max(), obs.index.max())
    model_trimmed = model.loc[start_date:end_date]
    obs_trimmed = obs.loc[start_date:end_date]
    model_cleaned, obs_cleaned = remove_empty_dates(model_trimmed, obs_trimmed)
    return model_cleaned, obs_cleaned

def clean_pair(model, obs):
    # Keep shared date window + drop NaN/Inf pairs
    model_t, obs_t = trim_series(model, obs)

    finite_mask = np.isfinite(model_t.values) & np.isfinite(obs_t.values)
    model_c = model_t.values[finite_mask]
    obs_c = obs_t.values[finite_mask]
    return model_c, obs_c

def validate_pair(model_c, obs_c):
    if model_c.size < 2:
        return False, 'Not enough valid points for regression'
    if np.nanstd(obs_c) == 0:
        return False, 'Observation series has zero variance'
    if np.nanmax(obs_c) == np.nanmin(obs_c):
        return False, 'Observation range is zero (NRMSE range undefined)'
    return True, None

def compute_stats(model, obs):
    """
    Compute stats all in one go
    """
    # First check that both inputs are not empty
    if model.empty or obs.empty:
        print('No data points, skipping stats computation')
        return None
    # print(f'Starting n points: model={len(model)}, obs={len(obs)}')
    model_c, obs_c = clean_pair(model, obs)
    # print(f'Valid finite pairs after cleaning: {len(model_c)}')
    ok, msg = validate_pair(model_c, obs_c)
    if not ok:
        print(msg)
        return None
    r, r_squared, n_points, fit_eq, m = calc_lrm_stats(model_c, obs_c)
    kge = calculate_kge(r, model_c, obs_c)
    rmse = calculate_rmse(model_c, obs_c)
    # nRMSE (normalized by the range of the observations)
    nrmse = calculate_nrmse(model_c, obs_c, normalization_factor='range')
    mae = calculate_mae(model_c, obs_c)
    return n_points, r, r_squared, fit_eq, m, kge, rmse, nrmse, mae

def format_stats_text(n, r, r_squared, fit_eq, kge, rmse, nrmse, mae):
    stats_text = '\n'.join([
    f'n = {n}',
    f'r = {r:.2f}',
    f'R² = {r_squared:.2f}',
    f'  {fit_eq}',
    f'RMSE = {rmse:.2f} m',
    f'NRMSE (range) = {nrmse:.2f}',
    f'MAE = {mae:.2f} m',
    f'KGE = {kge:.2f}',
    ])
    return stats_text

def dowy_from_index(index):
    """Convert a DatetimeIndex to day-of-water-year (Oct 1 = 1)."""
    doy = index.dayofyear.values
    dowy = doy.copy()
    dowy[doy >= 274] = doy[doy >= 274] - 273  # Oct 1 – Dec 31 inclusive
    dowy[doy < 274]  = doy[doy < 274]  + 92   # Jan 1 – Sep 30 inclusive
    return dowy

def add_dowy_colorbar(fig, sc, ax):
    """Add a Day-of-Water-Year colorbar to ax."""
    cbar = fig.colorbar(sc, ax=ax)
    cbar.set_label('Day of Water Year')
    cbar.set_ticks([1, 32, 61, 92, 122, 153, 183, 214, 245, 275, 306, 336])
    cbar.set_ticklabels(['Oct', 'Nov', 'Dec', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep'])

### NSE/KGE for daily modeled vs. SNOTEL depth


In [ ]:
basin = 'yampa'
thisvar = 'thickness'
unified_df = prep_isnobal_dfs(basin, thisvar, runnames=['unified'])[0]
snotel_df = process_snotel_por(df=unified_df)
plot_snotel_comparison(basin, thisvar, snotel_df,
                       isnobal_dfs = [unified_df],
                       runnames = ['unified'],
                       colors = ['dodgerblue'])

### PERIOD OF RECORD plot

In [ ]:
# PERIOD OF RECORD
# Drop the hours from the datetime index, keeping only the date
unified_df.index = unified_df.index.normalize()

# Loop through each of the snotel sites (columns)
for site in unified_df.columns:
    # Subset the SNOTEL data for this site (parse by site number in column name)
    unified_site_df = unified_df[site]
    snotel_site_df = snotel_df[[c for c in snotel_df.columns if site in c]]
    # Trim the snotel observation df to the unified date range
    snotel_site_subdf = snotel_site_df[(snotel_site_df.index >= unified_df.index.min()) & (snotel_site_df.index <= unified_df.index.max())]

    # Compute dowy
    dowy = dowy_from_index(unified_df.index)

    # Compute statistics
    stats = compute_stats(unified_site_df, np.squeeze(snotel_site_subdf/100, axis=1))
    if stats is None:
        print(f"Error computing stats for site {site}")
        continue

    # Scatterplot obs vs. modeled, modifying for cm to m plotting
    fig = plt.figure(figsize=(8, 5))
    sc = plt.scatter(snotel_site_subdf/100, unified_site_df, c=dowy, cmap='magma', label=site, s=20)

    # Add stats to the plot, modified from model_eval.ipynb
    n, r, r_squared, fit_eq, m, kge, rmse, nrmse, mae = stats
    stats_text = format_stats_text(n, r, r_squared, fit_eq, kge, rmse, nrmse, mae)
    plt.text(0.05, 0.90, stats_text, transform=plt.gca().transAxes, fontsize=10, verticalalignment='top')

    # Add 1:1 line
    max_val = max(np.nanmax(snotel_site_subdf/100), np.nanmax(unified_site_df))
    plt.plot([0, max_val], [0, max_val], color='lightgray', linestyle=':', lw=2)

    # Annotate the site name within the subplot
    plt.annotate(site, xy=(0.05, 0.925), xycoords='axes fraction', fontsize=11, fontstyle='italic')
    plt.xlabel('SNOTEL Snow Depth (m)')
    plt.ylabel('Modeled Snow Depth (m)')

    # Add colorbar for DOWY
    add_dowy_colorbar(fig, sc, plt.gca())
    plt.tight_layout()

### PERIOD OF RECORD plot, all sites in one figure with shared axes

In [ ]:
# PERIOD OF RECORD - ALL SITES IN ONE FIGURE - shared axes
# Drop the hours from the datetime index, keeping only the date
unified_df.index = unified_df.index.normalize()

sites = unified_df.columns
ncols = min(3, len(sites))
nrows = int(np.ceil(len(sites) / ncols))
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(6*ncols, 4*nrows), squeeze=False, sharex=True, sharey=True)

for ax, site in zip(axes.flat, sites):
    unified_site_df = unified_df[site]
    snotel_site_df = snotel_df[[c for c in snotel_df.columns if site in c]]
    snotel_site_subdf = snotel_site_df[(snotel_site_df.index >= unified_df.index.min()) & (snotel_site_df.index <= unified_df.index.max())]

    # Compute day of water year (Oct 1 = 1)
    dowy = dowy_from_index(unified_df.index)

    ax.annotate(site, xy=(0.05, 0.925), xycoords='axes fraction', fontsize=11, fontstyle='italic')
    ax.set_xlabel('SNOTEL Snow Depth (m)')
    ax.set_ylabel('Modeled Snow Depth (m)')

    # Compute statistics
    stats = compute_stats(unified_site_df, np.squeeze(snotel_site_subdf/100, axis=1))
    if stats is None:
        print(f"Error computing stats for site {site}")
        # Hide this subplot if stats can't be computed
        ax.axis('off')
        continue

    sc = ax.scatter(snotel_site_subdf/100, unified_site_df, c=dowy, cmap='magma', label=site, s=20)

    n, r, r_squared, fit_eq, m, kge, rmse, nrmse, mae = stats
    stats_text = format_stats_text(n, r, r_squared, fit_eq, kge, rmse, nrmse, mae)
    ax.text(0.05, 0.90, stats_text, transform=ax.transAxes, fontsize=10, verticalalignment='top')

    # Add colorbar for DOWY
    add_dowy_colorbar(fig, sc, ax)

for ax in axes.flat[:len(sites)]:
    if not ax.axison:
        continue
    # Plot the 1:1 line for the full range of the data
    max_val = 6
    ax.plot([0, max_val], [0, max_val], color='gray', linestyle=':', lw=2)

    # Force tick labels on all subplots (suppressed by sharex/sharey)
    ax.tick_params(axis='x', which='both', labelbottom=True)
    ax.tick_params(axis='y', which='both', labelleft=True)

# Hide unused subplots if site count doesn't fill the grid
for ax in axes.flat[len(sites):]:
    ax.axis('off')

plt.tight_layout()

### PERIOD OF RECORD plot, all sites in one figure with individual axes

In [ ]:
# PERIOD OF RECORD - ALL SITES IN ONE FIGURE - individual axes
# Drop the hours from the datetime index, keeping only the date
unified_df.index = unified_df.index.normalize()

sites = unified_df.columns
ncols = min(3, len(sites))
nrows = int(np.ceil(len(sites) / ncols))

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(6*ncols, 4*nrows), squeeze=False)

for ax, site in zip(axes.flatten(), sites):
    unified_site_df = unified_df[site]
    snotel_site_df = snotel_df[[c for c in snotel_df.columns if site in c]]
    snotel_site_subdf = snotel_site_df[(snotel_site_df.index >= unified_df.index.min()) & (snotel_site_df.index <= unified_df.index.max())]

    # Compute day of water year (Oct 1 = 1)
    dowy = dowy_from_index(unified_df.index)

    # Compute statistics
    stats = compute_stats(unified_site_df, np.squeeze(snotel_site_subdf/100, axis=1))
    if stats is None:
        print(f"Error computing stats for site {site}")
        # Hide this subplot if stats can't be computed
        ax.axis('off')
        continue

    sc = ax.scatter(snotel_site_subdf/100, unified_site_df, c=dowy, cmap='magma', label=site, s=20)

    n_points, r, r_squared, fit_eq, m, kge, rmse, nrmse, mae = stats
    stats_text = format_stats_text(n, r, r_squared, fit_eq, kge, rmse, nrmse, mae)
    ax.text(0.05, 0.90, stats_text, transform=ax.transAxes, fontsize=10, verticalalignment='top')

    ax.annotate(site, xy=(0.05, 0.925), xycoords='axes fraction', fontsize=11, fontstyle='italic')
    ax.set_xlabel('SNOTEL Snow Depth (m)')
    ax.set_ylabel('Modeled Snow Depth (m)')

    # Plot 1:1 line (only if stats were computed)
    max_val = max(np.nanmax(snotel_site_subdf/100), np.nanmax(unified_site_df))
    ax.plot([0, max_val], [0, max_val], color='gray', linestyle=':', lw=2)

    # Add colorbar for DOWY
    add_dowy_colorbar(fig, sc, ax)

# Hide unused subplots if site count doesn't fill the grid
for ax in axes.flat[len(sites):]:
    ax.axis('off')

plt.tight_layout()

### DIAGNOSTIC STATS and WATER YEAR plots, one figure per site (all WYs), one stats file for the basin

In [ ]:
# BY WATER YEAR
# Drop the hours from the datetime index, keeping only the date
unified_df.index = unified_df.index.normalize()
# Figure out the WYs based on unified_df september 30ths
sept_30s = unified_df.index[(unified_df.index.month == 9) & (unified_df.index.day == 30)]
water_years = sept_30s.year
# Set up a stats dictionary for storage by site and WY
stats_dict = {site: {} for site in unified_df.columns}
# Loop through each of the snotel sites (columns)
for site in unified_df.columns:
    print(site)
    # Set up the grid for this site
    ncols = min(3, len(water_years))
    nrows = int(np.ceil(len(water_years) / ncols))
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(6*ncols, 4*nrows), squeeze=False)
    axes = axes.flatten()

    for ax, wy in zip(axes, water_years):
        # get the min and max dates for this wy
        min_date = pd.Timestamp(year=wy-1, month=10, day=1)
        max_date = pd.Timestamp(year=wy, month=9, day=30)
        # Subset the SNOTEL data for this site (parse by site number in column name) and date range
        unified_site_df = unified_df[(unified_df.index >= min_date) & (unified_df.index <= max_date)][site]
        snotel_site_df = snotel_df[[c for c in snotel_df.columns if site in c]]
        # Trim the snotel observation df to the same date range
        snotel_site_subdf = snotel_site_df[(snotel_site_df.index >= min_date) & (snotel_site_df.index <= max_date)]

        # Compute day of water year (Oct 1 = 1)
        dowy = dowy_from_index(unified_site_df.index)

        # Compute statistics
        stats = compute_stats(unified_site_df, np.squeeze(snotel_site_subdf/100, axis=1))

        # Save these stats to a dictionary for later use if desired, e.g. stats_dict[site][wy] = stats
        # This will be None if stats cannot be computed
        stats_dict[site][wy] = stats

        if stats is None:
            print(f"Error computing stats for site {site} WY{wy}")
            # Maintain subplot but make axis invisible if stats can't be computed for visual consistency
            ax.axis('off')
            continue

        # Scatterplot obs vs. modeled, modifying for cm to m plotting
        # Change the marker color by DOWY (day of water year)
        sc = ax.scatter(snotel_site_subdf/100, unified_site_df, c=dowy, cmap='magma', label=site, s=20)

        # Annotate the site name within the subplot
        ax.annotate(f'{site} WY{wy}', xy=(0.05, 0.925), xycoords='axes fraction', fontsize=11, fontstyle='italic')
        ax.set_xlabel('SNOTEL Snow Depth (m)')
        ax.set_ylabel('Modeled Snow Depth (m)')

        n, r, r_squared, fit_eq, m, kge, rmse, nrmse, mae = stats
        # Add 1:1 line only if stats are successfully computed
        max_val = max(np.nanmax(snotel_site_subdf/100), np.nanmax(unified_site_df))
        ax.plot([0, max_val], [0, max_val], color='lightgray', linestyle=':', lw=2)

        # Add stats to the plot, modified from model_eval.ipynb
        stats_text = format_stats_text(n, r, r_squared, fit_eq, kge, rmse, nrmse, mae)
        ax.text(0.05, 0.90, stats_text, transform=ax.transAxes,
                fontsize=10, verticalalignment='top')

        # Add colorbar for DOWY
        add_dowy_colorbar(fig, sc, ax)

    # Turn off unused subplots
    for ax in axes[len(water_years):]:
        ax.set_visible(False)

    fig.suptitle(site)
    plt.tight_layout()

In [ ]:
# Save this to a csv with a multiindex of site and water year, and columns for each stat
stats_rows = []
for site, wy_dict in stats_dict.items():
    for wy, stats in wy_dict.items():
        if stats is None:
            stats_rows.append({'site': site, 'water_year': wy, 'n': None, 'r': None, 'r_squared': None,
                               'fit_eq': None, 'm': None, 'kge': None, 'rmse': None, 'nrmse': None, 'mae': None})
        else:
            n, r, r_squared, fit_eq, m, kge, rmse, nrmse, mae = stats
            stats_rows.append({'site': site, 'water_year': wy, 'n': n, 'r': r, 'r_squared': r_squared,
                               'fit_eq': fit_eq, 'm': m, 'kge': kge, 'rmse': rmse, 'nrmse': nrmse, 'mae': mae})
stats_df = pd.DataFrame(stats_rows)
stats_df.set_index(['site', 'water_year'], inplace=True)
stats_df

In [ ]:
outdir = '/uufs/chpc.utah.edu/common/home/skiles-group3/jmhu/thp/statistics'
# Date the output file with the current date and time for versioning
from datetime import datetime
timestamp = datetime.now().strftime('%Y%m%d')
df_name  = f'{basin}_isnobal_{thisvar}_stats_by_snotelsite_wy_{timestamp}.csv'
outname = f'{outdir}/{basin}/{df_name}'
print(f'Saving stats to {outname}')

# Create the parent directories if they don't exist
os.makedirs(os.path.dirname(outname), exist_ok=True)
stats_df.to_csv(outname)

# Also save a note on the df as a README.md for reference
note = f'{df_name} contains a multi-index dataframe of iSnobal (unified) statistics of {thisvar}, grouped by <SNOTEL site (sitenum)> and then <Water Year>.\n\nEach water year entry contains the following values <n, r, r_squared, fit_eq, m, kge, rmse, nrmse, mae> (or NaN/None if stats could not be computed due to insufficient data or zero variance in observations).\n\nRead back in using pd.read_csv({df_name}, index_col=["site", "water_year"]))'

readme_outname = f'{outdir}/{basin}/README.md'
print(f'Saving README note to {readme_outname}')
with open(readme_outname, 'w') as f:
    f.write(note)

In [ ]:
# Raise exception to stop the notebook here since this is the end of the intended workflow
raise Exception('End of workflow - stats computed and saved to csv')

# More plots

In [ ]:
# BY WATER YEAR - single plot
# Drop the hours from the datetime index, keeping only the date
unified_df.index = unified_df.index.normalize()
sites = unified_df.columns
# Figure out the WYs based on unified_df september 30ths
sept_30s = unified_df.index[(unified_df.index.month == 9) & (unified_df.index.day == 30)]
water_years = sept_30s.year

fig, axes = plt.subplots(nrows=len(sites), ncols=len(water_years),
                         figsize=(6*len(water_years), 4*len(sites)), squeeze=False)

# Loop through each of the snotel sites (columns)
for i, site in enumerate(sites):
    for j, wy in enumerate(water_years):
        print(site)
        ax = axes[i, j]
        # get the min and max dates for this wy
        min_date = pd.Timestamp(year=wy-1, month=10, day=1)
        max_date = pd.Timestamp(year=wy, month=9, day=30)

        # Subset the SNOTEL data for this site (parse by site number in column name) and date range
        unified_site_df = unified_df[(unified_df.index >= min_date) & (unified_df.index <= max_date)][site]
        snotel_site_df = snotel_df[[c for c in snotel_df.columns if site in c]]
        # Trim the snotel observation df to the same date range
        snotel_site_subdf = snotel_site_df[(snotel_site_df.index >= min_date) & (snotel_site_df.index <= max_date)]

        # Compute day of water year (Oct 1 = 1)
        dowy = dowy_from_index(unified_site_df.index)

        # Annotate the site name within the subplot
        ax.annotate(f'{site} WY{wy}', xy=(0.05, 0.925), xycoords='axes fraction', fontsize=11, fontstyle='italic')
        ax.set_xlabel('SNOTEL Snow Depth (m)')
        ax.set_ylabel('Modeled Snow Depth (m)')

        # Compute statistics
        stats = compute_stats(unified_site_df, np.squeeze(snotel_site_subdf/100, axis=1))
        if stats is None:
            print(f"Error computing stats for site {site} WY{wy}")
            # Maintain subplot but make axis invisible if stats can't be computed for visual consistency
            ax.axis('off')
            continue

        n, r, r_squared, fit_eq, m, kge, rmse, nrmse, mae = stats
        # Add 1:1 line only if stats are successfully computed
        max_val = max(np.nanmax(snotel_site_subdf/100), np.nanmax(unified_site_df))
        ax.plot([0, max_val], [0, max_val], color='lightgray', linestyle=':', lw=2)
        # Scatterplot obs vs. modeled, modifying for cm to m plotting
        # Change the marker color by DOWY (day of water year)
        sc = ax.scatter(snotel_site_subdf/100, unified_site_df, c=dowy, cmap='magma', label=site, s=20)

        # Add stats to the plot, modified from model_eval.ipynb
        stats_text = format_stats_text(n, r, r_squared, fit_eq, kge, rmse, nrmse, mae)

        # Slope-based placement
        if m > 1.2:
            x_txt, y_txt, ha, va = 0.95, 0.05, 'right', 'bottom'   # bottom-right
        else:
            x_txt, y_txt, ha, va = 0.05, 0.90, 'left', 'top'       # top-left

        ax.text(x_txt, y_txt, stats_text, transform=ax.transAxes,
                ha=ha, va=va, fontsize=10)

        # Add colorbar for DOWY
        add_dowy_colorbar(fig, sc, ax)

plt.tight_layout()

In [ ]:
# BY WATER YEAR - single plot - share xy across water years for each site
# Drop the hours from the datetime index, keeping only the date
unified_df.index = unified_df.index.normalize()
sites = unified_df.columns
# Figure out the WYs based on unified_df september 30ths
sept_30s = unified_df.index[(unified_df.index.month == 9) & (unified_df.index.day == 30)]
water_years = sept_30s.year

fig, axes = plt.subplots(nrows=len(sites), ncols=len(water_years),
                         figsize=(6*len(water_years), 4*len(sites)), squeeze=False, sharex=True, sharey=True)

# Loop through each of the snotel sites (columns)
for i, site in enumerate(sites):
    for j, wy in enumerate(water_years):
        print(site)
        ax = axes[i, j]
        # get the min and max dates for this wy
        min_date = pd.Timestamp(year=wy-1, month=10, day=1)
        max_date = pd.Timestamp(year=wy, month=9, day=30)

        # Subset the SNOTEL data for this site (parse by site number in column name) and date range
        unified_site_df = unified_df[(unified_df.index >= min_date) & (unified_df.index <= max_date)][site]
        snotel_site_df = snotel_df[[c for c in snotel_df.columns if site in c]]
        # Trim the snotel observation df to the same date range
        snotel_site_subdf = snotel_site_df[(snotel_site_df.index >= min_date) & (snotel_site_df.index <= max_date)]

        # Compute day of water year (Oct 1 = 1)
        dowy = dowy_from_index(unified_site_df.index)

        # Annotate the site name within the subplot
        ax.annotate(f'{site} WY{wy}', xy=(0.05, 0.925), xycoords='axes fraction', fontsize=11, fontstyle='italic')
        ax.set_xlabel('SNOTEL Snow Depth (m)')
        ax.set_ylabel('Modeled Snow Depth (m)')

        # Compute statistics
        stats = compute_stats(unified_site_df, np.squeeze(snotel_site_subdf/100, axis=1))
        if stats is None:
            print(f"Error computing stats for site {site} WY{wy}")
            # Maintain subplot but make axis invisible if stats can't be computed for visual consistency
            ax.axis('off')
            continue

        n, r, r_squared, fit_eq, m, kge, rmse, nrmse, mae = stats
        # Scatterplot obs vs. modeled, modifying for cm to m plotting
        # Change the marker color by DOWY (day of water year)
        sc = ax.scatter(snotel_site_subdf/100, unified_site_df, c=dowy, cmap='magma', label=site, s=20)

        # Add stats to the plot, modified from model_eval.ipynb
        stats_text = format_stats_text(n, r, r_squared, fit_eq, kge, rmse, nrmse, mae)

        # Slope-based placement
        if m > 1.2:
            x_txt, y_txt, ha, va = 0.95, 0.05, 'right', 'bottom'   # bottom-right
        else:
            x_txt, y_txt, ha, va = 0.05, 0.90, 'left', 'top'       # top-left

        ax.text(x_txt, y_txt, stats_text, transform=ax.transAxes,
                ha=ha, va=va, fontsize=10)

        # Add colorbar for DOWY
        add_dowy_colorbar(fig, sc, ax)

    # Loop through the axes and add 1:1 line only if the axis is on (i.e., stats were successfully computed)
    for j, wy in enumerate(water_years):
        ax = axes[i, j]
        if not ax.axison:
            continue
        max_val = 6
        ax.plot([0, max_val], [0, max_val], color='lightgray', linestyle=':', lw=2)

plt.tight_layout()

In [ ]:
# TODO save to individual plot file
# TODO save to giant subplots file
# Save the figure
# outdir = '/uufs/chpc.utah.edu/common/home/u6058223/public_html/thp_update'
# outname = f'{outdir}/{basin}/{basin}_snotel_por_{thisvar}_scatter.png'
# plt.savefig(f'{site}_por_snotel_comparison.png', dpi=300, bbox_inches='tight')
# print(f'Saving to {outname}')
# plt.savefig(outname, dpi=300, bbox_inches='tight')

In [ ]:
def site_to_filename(s):
    """
    Process sitename for filename use
    Example
    >>> site_to_filename('Cascade #2 (387)')
    'Cascade_2_387'
    """
    return '_'.join(re.findall(r'[A-Za-z0-9]+', s))

In [ ]:
# Now, split the stats up by DOWY ranges to get at seasonal differences
# Ranges should be quarterly to start
# OND early season accumulation
# JFM peak accumulation
# AMJ peak and ablation onset
# JAS summer (low to no snow)
# To keep things manageable, we should do this per site

# BY WATER YEAR - One figure per site
# WYs as columns, DOWY ranges as rows (shared axes)

# Drop the hours from the datetime index, keeping only the date
unified_df.index = unified_df.index.normalize()
sites = unified_df.columns

# Figure out the WYs based on unified_df september 30ths
sept_30s = unified_df.index[(unified_df.index.month == 9) & (unified_df.index.day == 30)]
water_years = sept_30s.year

dt_ranges = {
    'OND': ((10, 1), (12, 31)),   # Oct 1 - Dec 31
    'JFM': ((1, 1), (3, 31)),     # Jan 1 - Mar 31
    'AMJ': ((4, 1), (6, 30)),     # Apr 1 - Jun 30
    'JAS': ((7, 1), (9, 30)),     # Jul 1 - Sep 30
}

# Loop through each of the snotel sites (columns)
for site in sites:
    fig, axes = plt.subplots(ncols=len(dt_ranges), nrows=len(water_years),
                         figsize=(5*len(water_years), 2.5*len(dt_ranges)),
                         squeeze=False, sharex=True, sharey=True)
    fig.suptitle(site)
    # Loop WY
    for i, wy in enumerate(water_years):
        # Loop quarter
        for j, (dt_label, (dt_min, dt_max)) in enumerate(dt_ranges.items()):
            ax = axes[i, j]
            if i == 0:
                ax.set_title(dt_label)
            if j == 0:
                ax.annotate(f'WY {wy}', xy=(-0.25, 0.5), xycoords='axes fraction', fontsize=12, ha='right', va='center', rotation=90, fontstyle='italic')
            # get the min and max dates for this quarter
            if dt_min[0] >= 10:  # if the start month is Oct or later, it's in the previous calendar year
                 min_date = pd.Timestamp(year=wy-1, month=dt_min[0], day=dt_min[1])
                 max_date = pd.Timestamp(year=wy-1, month=dt_max[0], day=dt_max[1])
            else:
                min_date = pd.Timestamp(year=wy, month=dt_min[0], day=dt_min[1])
                max_date = pd.Timestamp(year=wy, month=dt_max[0], day=dt_max[1])

            # Subset the data for this site (parse by site number in column name) and date range
            unified_site_df = unified_df[(unified_df.index >= min_date) & (unified_df.index <= max_date)][site]
            snotel_site_df = snotel_df[[c for c in snotel_df.columns if site in c]]
            # Trim the snotel observation df to the same date range
            snotel_site_subdf = snotel_site_df[(snotel_site_df.index >= min_date) & (snotel_site_df.index <= max_date)]

            # Compute day of water year (Oct 1 = 1)
            dowy = dowy_from_index(unified_site_df.index)

            # Annotate the site name within the subplot
            # ax.annotate(f'{site} {dt_label} WY {wy}', xy=(0.05, 0.925), xycoords='axes fraction', fontsize=11, fontstyle='italic')
            ax.set_xlabel('SNOTEL Snow Depth (m)')
            ax.set_ylabel('Modeled Snow Depth (m)')

            # Compute statistics
            stats = compute_stats(unified_site_df, np.squeeze(snotel_site_subdf/100, axis=1))
            if stats is None:
                print(f"Error computing stats for site {site} WY{wy}")
                # Maintain subplot but make axis invisible if stats can't be computed for visual consistency
                ax.axis('off')
                continue

            n, r, r_squared, fit_eq, m, kge, rmse, nrmse, mae = stats
            # Scatterplot obs vs. modeled, modifying for cm to m plotting
            # Change the marker color by DOWY (day of water year)
            sc = ax.scatter(snotel_site_subdf/100, unified_site_df, c=dowy, cmap='magma', label=site, s=20)

            # Add stats to the plot, modified from model_eval.ipynb
            stats_text = format_stats_text(n, r, r_squared, fit_eq, kge, rmse, nrmse, mae)

            # Slope-based placement
            if m > 1.2:
                x_txt, y_txt, ha, va = 0.95, 0.05, 'right', 'bottom'   # bottom-right
            else:
                x_txt, y_txt, ha, va = 0.05, 0.975, 'left', 'top'       # top-left

            ax.text(x_txt, y_txt, stats_text, transform=ax.transAxes,
                    ha=ha, va=va, fontsize=10)

            # Add colorbar for DOWY
            add_dowy_colorbar(fig, sc, ax)

        # Loop through the axes and add 1:1 line only if the axis is on (i.e., stats were successfully computed)
        for j in range(len(dt_ranges)):
            ax = axes[i, j]
            if not ax.axison:
                continue
            max_val = 6
            ax.plot([0, max_val], [0, max_val], color='lightgray', linestyle=':', lw=2)
    plt.tight_layout()
    outdir = f'/uufs/chpc.utah.edu/common/home/u6058223/public_html/thp_update/individual'
    # Extract sitename and number from site (####) -> site_####
    sitename = site_to_filename(site)
    outname = f'{outdir}/{basin}/{sitename}_{basin}_snotel_por_quarters_{thisvar}_scatter.png'
    print(f'Saving to {outname}')
    plt.savefig(outname, dpi=300, bbox_inches='tight')

In [ ]:
# For Animas and Yampa, can plot (baseline vs. unified) albedo at SNOTEL for reference

In [ ]:
# Albedo sensitive days (stronger correlation with later DOWY/springtime)  vs. precip issues (potential correlation with depth increases? less correlated to DOWY than storm occurrence)

# Day check


In [ ]:
import xarray as xr

In [ ]:
basin = 'jordan'
dt = '20240315'
hour = '23'
wy = int(dt[:4]) + 1 if int(dt[4:6]) >= 10 else int(dt[:4])
# day_dir = f'/uufs/chpc.utah.edu/common/home/skiles-group2/model_runs_jmh/thp/{basin}_hrrr_radiation/wy{wy}/{basin}/run{dt}'
# original run
day_dir = f'/uufs/chpc.utah.edu/common/home/skiles-group2/model_runs_jmh/thp/{basin}_hrrr_radiation/wy{wy}/missing_data_jordan/{basin}/run{dt}'
print(day_dir)
out_fns = h.fn_list(day_dir, f'snow.nc')
print(out_fns)
# Open up all files at the timestamp hours
ds = xr.open_mfdataset(out_fns, combine='by_coords', compat='override').sel(time=f'{dt}T{hour}:00:00')
ds

In [ ]:
ds['thickness'].plot.imshow()

In [ ]:
# Quick day check
def check_day_outputs(basin, dt='20231001', hour='18'):
    '''hardcoded for thp runs, but can modify later'''
    # Compute wy directory from dt
    wy = int(dt[:4]) + 1 if int(dt[4:6]) >= 10 else int(dt[:4])
    day_dir = f'/uufs/chpc.utah.edu/common/home/skiles-group2/model_runs_jmh/thp/{basin}_hrrr_radiation/wy{wy}/{basin}/run{dt}'

    out_fns = h.fn_list(day_dir, f'*.nc')
    # Open up all files at the timestamp hours
    ds = xr.open_mfdataset(out_fns, combine='by_coords', compat='override').sel(time=f'{dt}T{hour}:00:00')
    # Drop the projection
    ds = ds.drop('projection')
    return ds

def plot_day_outputs(ds):
    # Plot it up
    for var in ds.data_vars:
        print(var)
        plt.figure()
        ds[var].plot.imshow()

ds = check_day_outputs(basin='jordan',dt='20231001', hour='23')
plot_day_outputs(ds)